In [2]:
import os
import io
import sys
import contextlib, io, traceback
from dotenv import load_dotenv
import subprocess

from openai import OpenAI
import anthropic
from huggingface_hub import login, InferenceClient
from transformers import AutoTokenizer
import gradio as gr

ModuleNotFoundError: No module named 'transformers'

In [2]:
# Config
load_dotenv(override=True)

OPENAI_MODEL = "gpt-4o"
CLAUDE_MODEL = "claude-3-5-sonnet-20240620"

openai = OpenAI()
claude = anthropic.Anthropic()

system_message = """
You are an assistant that reimplements Python code in high performance C++ for PC Win11.
Respond only with C++ code; use comments sparingly and do not provide any expplanation other then 
occasional comments. The C++ reponse needs to produce and identical output in the fastert possible time
"""

pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(100_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [3]:
# Utils
def user_prompt_for(python):
    user_prompt = f"""
    Rewrite this Python code in C++ with the fastest possible implementation that produces identical 
    output in the least time. Respond only with C++ code; do not explain your work other than a few comments.
    Pay attation to number types to ensure no int overflows. Remember to #include all necessary C++
    packages such as iomanip.

    {python}
    """
    return user_prompt


def messages_for(python):
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt_for(python)},
    ]


def write_output(cpp):
    code = cpp.replace("```cpp", "").replace("```", "")
    with open("optimized.cpp", "w") as f:
        f.write(code)


def optimize_gpt(python):
    stream = openai.chat.completions.create(
        model=OPENAI_MODEL,
        messages = messages_for(python),
        stream=True
    )
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        print(fragment, end="", flush=True)
    write_output(reply)
    return reply


def optimize_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            print(text, end="", flush=True)
    write_output(reply)
    return reply

In [4]:
exec(pi)

Result: 3.141592658589
Execution Time: 28.838212 seconds


In [5]:
optimize_gpt(pi)

```cpp
#include <iostream>
#include <iomanip>
#include <chrono>
#include <limits>

double calculate(int iterations, int param1, int param2) {
    double result = 1.0;
    for (int i = 1; i <= iterations; ++i) {
        double j = i * param1 - param2;
        result -= (1.0 / j);
        j = i * param1 + param2;
        result += (1.0 / j);
    }
    return result;
}

int main() {
    using namespace std::chrono;

    auto start_time = high_resolution_clock::now();
    double result = calculate(100000000, 4, 1) * 4;
    auto end_time = high_resolution_clock::now();

    auto exec_time = duration_cast<duration<double>>(end_time - start_time).count();

    std::cout << std::fixed << std::setprecision(12);
    std::cout << "Result: " << result << std::endl;
    std::cout << "Execution Time: " << exec_time << " seconds" << std::endl;

    return 0;
}
```

'```cpp\n#include <iostream>\n#include <iomanip>\n#include <chrono>\n#include <limits>\n\ndouble calculate(int iterations, int param1, int param2) {\n    double result = 1.0;\n    for (int i = 1; i <= iterations; ++i) {\n        double j = i * param1 - param2;\n        result -= (1.0 / j);\n        j = i * param1 + param2;\n        result += (1.0 / j);\n    }\n    return result;\n}\n\nint main() {\n    using namespace std::chrono;\n\n    auto start_time = high_resolution_clock::now();\n    double result = calculate(100000000, 4, 1) * 4;\n    auto end_time = high_resolution_clock::now();\n\n    auto exec_time = duration_cast<duration<double>>(end_time - start_time).count();\n\n    std::cout << std::fixed << std::setprecision(12);\n    std::cout << "Result: " << result << std::endl;\n    std::cout << "Execution Time: " << exec_time << " seconds" << std::endl;\n\n    return 0;\n}\n```'

In [6]:
# Compile and run C++ code on PC WIN11 WSL Ubuntu distribution using g++
!g++ -O3 -std=c++17 -o optimized optimized.cpp
!./optimized

Result: 3.141592658589
Execution Time: 0.208341279000 seconds


In [7]:
optimize_claude(pi)

#include <iostream>
#include <iomanip>
#include <chrono>

double calculate(int64_t iterations, int64_t param1, int64_t param2) {
    double result = 1.0;
    for (int64_t i = 1; i <= iterations; ++i) {
        int64_t j = i * param1 - param2;
        result -= 1.0 / j;
        j = i * param1 + param2;
        result += 1.0 / j;
    }
    return result;
}

int main() {
    auto start = std::chrono::high_resolution_clock::now();
    double result = calculate(100'000'000, 4, 1) * 4;
    auto end = std::chrono::high_resolution_clock::now();

    std::cout << std::fixed << std::setprecision(12);
    std::cout << "Result: " << result << std::endl;

    std::chrono::duration<double> elapsed = end - start;
    std::cout << "Execution Time: " << std::fixed << std::setprecision(6) << elapsed.count() << " seconds" << std::endl;

    return 0;
}

'#include <iostream>\n#include <iomanip>\n#include <chrono>\n\ndouble calculate(int64_t iterations, int64_t param1, int64_t param2) {\n    double result = 1.0;\n    for (int64_t i = 1; i <= iterations; ++i) {\n        int64_t j = i * param1 - param2;\n        result -= 1.0 / j;\n        j = i * param1 + param2;\n        result += 1.0 / j;\n    }\n    return result;\n}\n\nint main() {\n    auto start = std::chrono::high_resolution_clock::now();\n    double result = calculate(100\'000\'000, 4, 1) * 4;\n    auto end = std::chrono::high_resolution_clock::now();\n\n    std::cout << std::fixed << std::setprecision(12);\n    std::cout << "Result: " << result << std::endl;\n\n    std::chrono::duration<double> elapsed = end - start;\n    std::cout << "Execution Time: " << std::fixed << std::setprecision(6) << elapsed.count() << " seconds" << std::endl;\n\n    return 0;\n}'

In [8]:
# Compile and run C++ code on PC WIN11 WSL Ubuntu distribution using g++
!g++ -O3 -std=c++17 -o optimized optimized.cpp
!./optimized

Result: 3.141592658589
Execution Time: 0.205400 seconds


In [9]:
def stream_gpt(python):
    stream = openai.chat.completions.create(
        model=OPENAI_MODEL,
        messages = messages_for(python),
        stream=True
    )
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply.replace("```cpp", "").replace("```", "")


def stream_claude(python):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(python)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply.replace("```cpp", "").replace("```", "")


def optimize(python, model):
    if model == "GPT":
        result = stream_gpt(python)
    elif model == "Claude":
        result = stream_claude(python)
    else:
        raise NotImplementedError("Model not supported")
    
    for stream_so_far in result:
        yield stream_so_far



def execute_python(code: str) -> str:
    """
    Execute Python source in an isolated namespace.
    Returns only captured stdout (like original), never leaves sys.stdout redirected.
    """
    stdout_buf = io.StringIO()
    ns = {}          # empty globals/locals so user code can't modify notebook globals
    try:
        compiled = compile(code, "<user_code>", "exec")
        with contextlib.redirect_stdout(stdout_buf):
            exec(compiled, ns, ns)
    except Exception:
        # Append traceback to stdout (mimics original behavior of letting exceptions surface visibly)
        import traceback as _tb
        stdout_buf.write(_tb.format_exc())
    return stdout_buf.getvalue()


def execute_cpp(code: str) -> str:
    """
    Write C++ code to file, compile and execute it using WSL Ubuntu g++.
    Returns stdout output. Raises CalledProcessError on failure.
    """
    write_output(code)
    try:
        compile_cmd = ["g++", "-O3", "-std=c++17", "-o", "optimized", "optimized.cpp"]
        run_cmd = ["./optimized"]
        subprocess.run(compile_cmd, check=True, text=True, capture_output=True)
        result = subprocess.run(run_cmd, check=True, text=True, capture_output=True)
        return result.stdout
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"An error occurred:\n{e.stderr}")

In [10]:
css = """
.python {background-color: #306998;}
.cpp {background-color: #228B22;}
"""

In [11]:
with gr.Blocks(css=css) as ui:
    gr.Markdown("## Convert code from Python to C++")
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=10, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=10)
    with gr.Row():
        model = gr.Dropdown(
            label="Select Model",
            choices=["GPT", "Claude"], 
            value="GPT"
        )
        convert = gr.Button("Convert code")
        clear = gr.Button("Clear")
    with gr.Row():
        python_run = gr.Button("Run Python")
        cpp_run = gr.Button("Run C++")
    with gr.Row():
        python_out = gr.TextArea(label="Python result:", elem_classes=["python"])
        cpp_out = gr.TextArea(label="C++ result:", elem_classes=["cpp"])

    convert.click(optimize, inputs=[python, model], outputs=cpp)
    clear.click(lambda: ("", ""), inputs=None, outputs=[python, cpp])
    python_run.click(execute_python, inputs=[python], outputs=[python_out])
    cpp_run.click(execute_cpp, inputs=[cpp], outputs=[cpp_out])


ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
